# GI Stage 2 — Data Processing Notebook
### GDSC2/CCLE realigned to GIBridge-MLP's GI-specific gene space - Warm-start + Cell-cold splits only

**What changed vs. OncoBridge's Stage 2 data pipeline:**

| | OncoBridge (pan-cancer) | GI Stage 2 (this notebook) |
|---|---|---|
| Gene space | OncoBridge-MMCAT's 8011/3500/2500/6000 genes | GIBridge-MLP's own 4000/1750/1250/3000 genes |
| Cell lines | All 388 (any cancer type) | 79, restricted to 4 GI lineages (Bowel, Esophagus/Stomach, Pancreas, Liver) |
| Drugs | All 74 (>=50 pan-cancer samples) | Whatever of those 74 has GI-cell-line coverage - no re-filtering by a GI-specific minimum |
| Splits | Warm / cell-cold / drug-cold (scaffold) | Warm / cell-cold only - too few GI drugs for a meaningful drug-cold holdout |
| Scaling | Fit fresh on CCLE training cell lines | Same - fit fresh on GI CCLE training cell lines |

**Correction from earlier discussion:** the fitted scalers saved from GIBridge Stage 1 are **not**
reused here. Checking the original OncoBridge pipeline (`data-analysis.ipynb` Cell 14) confirmed the
established, working precedent: CCLE data is scaled with its own freshly-fit scaler (same scaler type
per modality - StandardScaler for mRNA, MaxAbsScaler for CNV - but fit on CCLE statistics, not TCGA
statistics), because cell lines and tissue samples have systematically different baseline expression
distributions unrelated to biology. What actually transfers from GIBridge Stage 1 is the **selected
gene names only**, so the frozen encoder receives the correct genes in the correct order.

**Dropped cell lines:** Biliary Tract (Intraductal Papillary Neoplasm of the Bile Duct) - only 1 cell
line, unusable for any split.

**Inputs needed:**
1. GIBridge's 4 saved gene-name JSONs (from the Stage 1 addition cell)
2. `drug-cell-line-data-phase-2` - raw CCLE omics CSVs + Model.csv (same Kaggle dataset OncoBridge used)
3. Your GDSC2 splits dataset - for the pre-merged IC50 table (`split_warm_*.parquet`) and precomputed
   Morgan fingerprints


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, time, gc, warnings
from sklearn.preprocessing import StandardScaler, MaxAbsScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

def section(title):
    print('\n' + '=' * 65)
    print(f'  {title}')
    print('=' * 65)

def mem_usage(df, name):
    mb = df.memory_usage(deep=True).sum() / 1e6
    print(f'  [{name}] RAM usage: {mb:.1f} MB | shape: {df.shape}')

print('Setup complete.')


Setup complete.


In [2]:
CFG = dict(
    SEED = 42,

    # -- Split fractions (warm-start + cell-cold only; no drug-cold) --------
    WARM_VAL_FRAC   = 0.10,
    WARM_TEST_FRAC  = 0.10,
    CELL_COLD_FRAC  = 0.20,   # -- adjust after seeing per-lineage counts printed below

    # -- GI lineages to keep (Biliary Tract dropped -- only 1 cell line) ----
    GI_LINEAGES_KEEP = ['Bowel', 'Esophagus/Stomach', 'Pancreas', 'Liver'],

    # -- Paths -- UPDATE these to your actual Kaggle dataset slugs ----------
    GENE_JSON_DIR = '/kaggle/input/datasets/proutkarshtiwari/selected-gene-json-gi-cancer',
    RAW_CCLE_BASE = '/kaggle/input/datasets/proutkarshtiwari/drug-cell-line-data-phase-2',
    GDSC2_BASE    = '/kaggle/input/datasets/proutkarshtiwari/smiles-correct-data-gdsc2',
    MODEL_CSV     = 'Model.csv',
    MRNA_CSV      = 'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
    CNV_CSV       = 'OmicsCNGeneWGS.csv',
    MUT_CSV       = 'OmicsSomaticMutationsMatrixDamaging.csv',
    METH_CSV      = 'CCLE_RRBS_TSS_1kb_20180614.txt',
    OUT_DIR       = '/kaggle/working/',
)

print('CFG loaded.')
print(f'GI lineages kept: {CFG["GI_LINEAGES_KEEP"]}')
print(f'Cell-cold fraction: {CFG["CELL_COLD_FRAC"]*100:.0f}% (review per-lineage counts before finalising)')


CFG loaded.
GI lineages kept: ['Bowel', 'Esophagus/Stomach', 'Pancreas', 'Liver']
Cell-cold fraction: 20% (review per-lineage counts before finalising)


In [3]:
section('GIBridge-MLP selected gene names')

GENES_GI = {}
for mod in ['mrna', 'cnv', 'mut', 'meth']:
    path = f"{CFG['GENE_JSON_DIR']}/gibridge_selected_genes_{mod}.json"
    with open(path) as f:
        GENES_GI[mod] = json.load(f)

print('Gene counts (must match GIBridge Stage 1 CONFIG exactly):')
for k, v in GENES_GI.items():
    print(f'  {k:>4}: {len(v):>6,} genes')
print(f'  Total features: {sum(len(v) for v in GENES_GI.values()):,}')



  GIBridge-MLP selected gene names
Gene counts (must match GIBridge Stage 1 CONFIG exactly):
  mrna:  4,000 genes
   cnv:  1,750 genes
   mut:  1,250 genes
  meth:  3,000 genes
  Total features: 10,000


In [4]:
section('IC50 pairs -- load full cohort and filter to GI lineages')

warm_tr = pd.read_parquet(f"{CFG['GDSC2_BASE']}/split_warm_tr.parquet")
warm_vl = pd.read_parquet(f"{CFG['GDSC2_BASE']}/split_warm_vl.parquet")
warm_te = pd.read_parquet(f"{CFG['GDSC2_BASE']}/split_warm_te.parquet")
ic50_all = pd.concat([warm_tr, warm_vl, warm_te], ignore_index=True)
del warm_tr, warm_vl, warm_te

print(f'Full cohort: {len(ic50_all):,} pairs | {ic50_all["ModelID"].nunique()} cell lines | '
      f'{ic50_all["DRUG_NAME"].nunique()} drugs')

ic50 = ic50_all[ic50_all['OncotreeLineage'].isin(CFG['GI_LINEAGES_KEEP'])].copy()
del ic50_all

print(f'\nAfter GI-lineage filter: {len(ic50):,} pairs | '
      f'{ic50["ModelID"].nunique()} cell lines | {ic50["DRUG_NAME"].nunique()} drugs')

print('\nCell lines per lineage:')
print(ic50.drop_duplicates('ModelID')['OncotreeLineage'].value_counts().to_string())

print('\nNo per-drug minimum re-applied within the GI subset -- the clinically standard')
print('GI drugs (5-FU, oxaliplatin) have too few pairs to survive a strict filter.')
print('Every drug the model sees is real GDSC2 data; the honesty burden shifts to')
print('reporting per-drug sample counts at evaluation time, not to filtering here.')

valid_model_ids = set(ic50['ModelID'].unique())
print(f'\nvalid_model_ids: {len(valid_model_ids)} GI cell lines')



  IC50 pairs -- load full cohort and filter to GI lineages
Full cohort: 9,967 pairs | 388 cell lines | 74 drugs

After GI-lineage filter: 1,579 pairs | 79 cell lines | 71 drugs

Cell lines per lineage:
OncotreeLineage
Esophagus/Stomach    30
Bowel                24
Pancreas             14
Liver                11

No per-drug minimum re-applied within the GI subset -- the clinically standard
GI drugs (5-FU, oxaliplatin) have too few pairs to survive a strict filter.
Every drug the model sees is real GDSC2 data; the honesty burden shifts to
reporting per-drug sample counts at evaluation time, not to filtering here.

valid_model_ids: 79 GI cell lines


In [5]:
section('CCLE mRNA -- reindexed to GIBridge gene space, filtered to GI cell lines')

mrna_raw = pd.read_csv(f"{CFG['RAW_CCLE_BASE']}/{CFG['MRNA_CSV']}", index_col=0, low_memory=False)
if 'IsDefaultEntryForModel' in mrna_raw.columns:
    mrna_raw = mrna_raw[mrna_raw['IsDefaultEntryForModel'] == 'Yes'].copy()
    mrna_raw = mrna_raw.set_index('ModelID')
    mrna_raw = mrna_raw.drop(columns=['SequencingID', 'ModelConditionID',
                                       'IsDefaultEntryForMC', 'IsDefaultEntryForModel'],
                              errors='ignore')

mrna_raw.columns = [str(c).split(' (')[0].strip() for c in mrna_raw.columns]
mrna_raw = mrna_raw.loc[:, ~mrna_raw.columns.duplicated(keep='first')]
mrna_raw = mrna_raw[mrna_raw.index.isin(valid_model_ids)]

overlap_m = set(mrna_raw.columns) & set(GENES_GI['mrna'])
print(f'mRNA overlap with GIBridge genes: {len(overlap_m)}/{len(GENES_GI["mrna"])}')

mrna_df = mrna_raw.reindex(columns=GENES_GI['mrna'], fill_value=0.0).fillna(0.0).astype(np.float32)
print(f'Final mRNA: {mrna_df.shape}')
valid_model_ids_mrna = set(mrna_df.index)
del mrna_raw; gc.collect()
mem_usage(mrna_df, 'mRNA')



  CCLE mRNA -- reindexed to GIBridge gene space, filtered to GI cell lines
mRNA overlap with GIBridge genes: 3747/4000
Final mRNA: (79, 4000)
  [mRNA] RAM usage: 1.3 MB | shape: (79, 4000)


In [6]:
section('CCLE CNV -- reindexed to GIBridge gene space')

cnv_raw = pd.read_csv(f"{CFG['RAW_CCLE_BASE']}/{CFG['CNV_CSV']}", index_col=0, low_memory=False)
if 'IsDefaultEntryForModel' in cnv_raw.columns:
    cnv_raw = cnv_raw[cnv_raw['IsDefaultEntryForModel'] == 'Yes'].copy()
    cnv_raw = cnv_raw.set_index('ModelID')
    cnv_raw = cnv_raw.drop(columns=['SequencingID', 'ModelConditionID',
                                     'IsDefaultEntryForMC', 'IsDefaultEntryForModel'],
                            errors='ignore')

cnv_raw.columns = [str(c).split(' (')[0].strip() for c in cnv_raw.columns]
cnv_raw = cnv_raw.loc[:, ~cnv_raw.columns.duplicated(keep='first')]
cnv_raw = cnv_raw[cnv_raw.index.isin(valid_model_ids)]
cnv_raw = cnv_raw.fillna(2.0).clip(lower=0.01)
cnv_raw = np.log2(cnv_raw / 2.0).astype(np.float32)

overlap_c = set(cnv_raw.columns) & set(GENES_GI['cnv'])
print(f'CNV overlap with GIBridge genes: {len(overlap_c)}/{len(GENES_GI["cnv"])}')

cnv_df = cnv_raw.reindex(columns=GENES_GI['cnv'], fill_value=0.0).astype(np.float32)
print(f'Final CNV: {cnv_df.shape}')
valid_model_ids_cnv = set(cnv_df.index)
del cnv_raw; gc.collect()
mem_usage(cnv_df, 'CNV')



  CCLE CNV -- reindexed to GIBridge gene space
CNV overlap with GIBridge genes: 1616/1750
Final CNV: (79, 1750)
  [CNV] RAM usage: 0.6 MB | shape: (79, 1750)


In [7]:
section('CCLE Mutation -- reindexed to GIBridge gene space')

mut_raw = pd.read_csv(f"{CFG['RAW_CCLE_BASE']}/{CFG['MUT_CSV']}", index_col=0, low_memory=False)
if 'IsDefaultEntryForModel' in mut_raw.columns:
    mut_raw = mut_raw[mut_raw['IsDefaultEntryForModel'] == 'Yes'].copy()
    mut_raw = mut_raw.set_index('ModelID')
    mut_raw = mut_raw.drop(columns=['SequencingID', 'ModelConditionID',
                                     'IsDefaultEntryForMC', 'IsDefaultEntryForModel'],
                            errors='ignore')

mut_raw.columns = [str(c).split(' (')[0].strip() for c in mut_raw.columns]
mut_raw = mut_raw.loc[:, ~mut_raw.columns.duplicated(keep='first')]
mut_raw = mut_raw[mut_raw.index.isin(valid_model_ids)]
mut_raw = mut_raw.fillna(0).clip(0, 1).round(0).astype(np.int8)

overlap_u = set(mut_raw.columns) & set(GENES_GI['mut'])
sparsity  = (mut_raw == 0).values.mean()
print(f'Mutation overlap with GIBridge genes: {len(overlap_u)}/{len(GENES_GI["mut"])}')
print(f'Sparsity (wildtype fraction): {sparsity*100:.1f}%')

mut_df = mut_raw.reindex(columns=GENES_GI['mut'], fill_value=0).astype(np.float32)
print(f'Final Mutation: {mut_df.shape}')
valid_model_ids_mut = set(mut_df.index)
del mut_raw; gc.collect()
mem_usage(mut_df, 'Mutation')



  CCLE Mutation -- reindexed to GIBridge gene space
Mutation overlap with GIBridge genes: 1198/1250
Sparsity (wildtype fraction): 99.3%
Final Mutation: (79, 1250)
  [Mutation] RAM usage: 0.4 MB | shape: (79, 1250)


In [8]:
section('CCLE Methylation (RRBS) -- reindexed to GIBridge gene space')

print('Loading CCLE_RRBS_TSS_1kb_20180614.txt...')
meth_raw = pd.read_csv(f"{CFG['RAW_CCLE_BASE']}/{CFG['METH_CSV']}", sep='\t', low_memory=False)
print(f'Raw shape: {meth_raw.shape}')

if 'gene' in meth_raw.columns:
    gene_col_values = meth_raw['gene'].values
else:
    gene_col_values = meth_raw.iloc[:, 0].values

meta_cols = ['gene', 'chr', 'fpos', 'tpos', 'strand', 'avg_coverage']
meth_data = meth_raw.drop(columns=meta_cols, errors='ignore')
meth_data = meth_data.apply(pd.to_numeric, errors='coerce').astype(np.float32)
meth_data.index = gene_col_values

n_tss_before = len(meth_data)
meth_data = meth_data.groupby(meth_data.index).mean()
print(f'TSS aggregation: {n_tss_before:,} TSS -> {len(meth_data):,} unique genes')

meth_data = meth_data.T

model_map = pd.read_csv(f"{CFG['RAW_CCLE_BASE']}/{CFG['MODEL_CSV']}", usecols=['ModelID', 'CCLEName'])
model_map = model_map.dropna(subset=['CCLEName'])
ccle_to_model = dict(zip(model_map['CCLEName'], model_map['ModelID']))

meth_data.index = meth_data.index.map(lambda x: ccle_to_model.get(x, None))
meth_data = meth_data[meth_data.index.notna()].copy()
meth_data.index = meth_data.index.astype(str)

meth_data = meth_data[meth_data.index.isin(valid_model_ids)].copy()
print(f'Filtered to GI cell lines: {len(meth_data)}')

gene_means_fill = meth_data.mean(axis=0)
meth_data = meth_data.fillna(gene_means_fill).fillna(0.5).clip(0.0, 1.0).astype(np.float32)

overlap_e = set(meth_data.columns) & set(GENES_GI['meth'])
print(f'Methylation overlap with GIBridge genes: {len(overlap_e)}/{len(GENES_GI["meth"])}')

meth_df = meth_data.reindex(columns=GENES_GI['meth'], fill_value=0.5).astype(np.float32)
print(f'Final Methylation: {meth_df.shape}')

valid_model_ids_meth = set(meth_df.index)
del meth_raw, meth_data, model_map, ccle_to_model
gc.collect()
mem_usage(meth_df, 'Methylation')



  CCLE Methylation (RRBS) -- reindexed to GIBridge gene space
Loading CCLE_RRBS_TSS_1kb_20180614.txt...
Raw shape: (20192, 850)
TSS aggregation: 20,192 TSS -> 16,494 unique genes
Filtered to GI cell lines: 67
Methylation overlap with GIBridge genes: 1511/3000
Final Methylation: (67, 3000)
  [Methylation] RAM usage: 0.8 MB | shape: (67, 3000)


In [9]:
section('Master alignment (GI subset)')

print('Cell lines per modality:')
print(f'  mRNA : {len(valid_model_ids_mrna):,}')
print(f'  CNV  : {len(valid_model_ids_cnv):,}')
print(f'  Mut  : {len(valid_model_ids_mut):,}')
print(f'  Meth : {len(valid_model_ids_meth):,}')

have_3 = valid_model_ids_mrna & valid_model_ids_cnv & valid_model_ids_mut
have_all4 = have_3 & valid_model_ids_meth
have_3_no_meth = have_3 - valid_model_ids_meth

print(f'\nAll 4 modalities: {len(have_all4):,}')
print(f'mRNA+CNV+Mut only: {len(have_3_no_meth):,} (methylation will be imputed with 0.5)')

if len(have_3_no_meth) > 0:
    imp_meth = pd.DataFrame(0.5, index=sorted(have_3_no_meth),
                             columns=GENES_GI['meth'], dtype=np.float32)
    meth_df = pd.concat([meth_df, imp_meth])

common_final = sorted(have_all4 | have_3_no_meth)
print(f'\nFinal usable GI cell lines: {len(common_final)}')

mrna_df = mrna_df.loc[mrna_df.index.isin(common_final)].reindex(common_final)
cnv_df  = cnv_df.loc[cnv_df.index.isin(common_final)].reindex(common_final)
mut_df  = mut_df.loc[mut_df.index.isin(common_final)].reindex(common_final)
meth_df = meth_df.loc[meth_df.index.isin(common_final)].reindex(common_final)

assert (mrna_df.index == cnv_df.index).all()
assert (mrna_df.index == mut_df.index).all()
assert (mrna_df.index == meth_df.index).all()
print('All modalities aligned OK')

ic50 = ic50[ic50['ModelID'].isin(common_final)].copy()
print(f'IC50 pairs after alignment: {len(ic50):,}')
print('\nCell lines per lineage (final, post-alignment):')
print(ic50.drop_duplicates('ModelID')['OncotreeLineage'].value_counts().to_string())



  Master alignment (GI subset)
Cell lines per modality:
  mRNA : 79
  CNV  : 79
  Mut  : 79
  Meth : 67

All 4 modalities: 67
mRNA+CNV+Mut only: 12 (methylation will be imputed with 0.5)

Final usable GI cell lines: 79
All modalities aligned OK
IC50 pairs after alignment: 1,579

Cell lines per lineage (final, post-alignment):
OncotreeLineage
Esophagus/Stomach    30
Bowel                24
Pancreas             14
Liver                11


In [10]:
section('Generating splits: warm-start + cell-cold (no drug-cold)')

np.random.seed(CFG['SEED'])
all_cls_arr = np.array(common_final)

cl_lineage_map = ic50.drop_duplicates('ModelID').set_index('ModelID')['OncotreeLineage'].to_dict()
lineage_arr = np.array([cl_lineage_map.get(cl, 'Unknown') for cl in all_cls_arr])

def safe_stratify_encode(labels):
    s = pd.Series(labels)
    counts = s.value_counts()
    dominant = counts.index[0]
    safe = np.where(s.map(counts) < 4, dominant, labels)
    le = LabelEncoder()
    return le.fit_transform(safe)

lineage_enc = safe_stratify_encode(lineage_arr)

# -- SPLIT 1: WARM-START (80/10/10 random pair split) -----------------------
print('-' * 60)
print('SPLIT 1: WARM-START (80/10/10 random pair split)')
print('-' * 60)

tv_pairs, ic50_warm_te = train_test_split(
    ic50, test_size=CFG['WARM_TEST_FRAC'], random_state=CFG['SEED'])
ic50_warm_tr, ic50_warm_vl = train_test_split(
    tv_pairs, test_size=CFG['WARM_VAL_FRAC'] / (1.0 - CFG['WARM_TEST_FRAC']),
    random_state=CFG['SEED'])

print(f'  Train pairs: {len(ic50_warm_tr):>5,} ({len(ic50_warm_tr)/len(ic50)*100:.1f}%)')
print(f'  Val   pairs: {len(ic50_warm_vl):>5,} ({len(ic50_warm_vl)/len(ic50)*100:.1f}%)')
print(f'  Test  pairs: {len(ic50_warm_te):>5,} ({len(ic50_warm_te)/len(ic50)*100:.1f}%)')

# -- SPLIT 2: CELL-COLD-START (stratified by GI lineage) --------------------
print('\n' + '-' * 60)
print(f'SPLIT 2: CELL-COLD-START ({CFG["CELL_COLD_FRAC"]*100:.0f}% cell lines held out, stratified by lineage)')
print('-' * 60)

sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=CFG['CELL_COLD_FRAC'],
                                    random_state=CFG['SEED'] + 1)
cell_tv_idx, cell_te_idx = next(sss_outer.split(all_cls_arr, lineage_enc))

cl_cell_trainval = list(all_cls_arr[cell_tv_idx])
cl_cell_test     = list(all_cls_arr[cell_te_idx])

tv_arr = np.array(cl_cell_trainval)
lineage_enc_tv = safe_stratify_encode(lineage_enc[cell_tv_idx])

sss_inner = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=CFG['SEED'])
tv_tr_idx, tv_vl_idx = next(sss_inner.split(tv_arr, lineage_enc_tv))

cl_cell_train = list(tv_arr[tv_tr_idx])
cl_cell_val   = list(tv_arr[tv_vl_idx])

assert not (set(cl_cell_train) & set(cl_cell_test)), 'CELL-COLD TRAIN/TEST LEAKAGE'
assert not (set(cl_cell_val) & set(cl_cell_test)), 'CELL-COLD VAL/TEST LEAKAGE'
assert not (set(cl_cell_train) & set(cl_cell_val)), 'CELL-COLD TRAIN/VAL OVERLAP'

ic50_cc_tr = ic50[ic50['ModelID'].isin(cl_cell_train)].copy()
ic50_cc_vl = ic50[ic50['ModelID'].isin(cl_cell_val)].copy()
ic50_cc_te = ic50[ic50['ModelID'].isin(cl_cell_test)].copy()

print(f'  Train cell lines: {len(cl_cell_train):>3} | pairs: {len(ic50_cc_tr):>5,}')
print(f'  Val   cell lines: {len(cl_cell_val):>3} | pairs: {len(ic50_cc_vl):>5,}')
print(f'  Test  cell lines: {len(cl_cell_test):>3} | pairs: {len(ic50_cc_te):>5,} (unseen)')
print(f'  Leakage assertions passed')

print('\nCell-cold TEST cell lines per lineage (review before finalising CELL_COLD_FRAC):')
cc_te_lineages = pd.Series([cl_lineage_map[c] for c in cl_cell_test])
print(cc_te_lineages.value_counts().to_string())
print('\nIf any lineage above has 0-1 test cell lines, consider raising CELL_COLD_FRAC')
print('or accepting that lineage will have a very noisy cell-cold estimate.')



  Generating splits: warm-start + cell-cold (no drug-cold)
------------------------------------------------------------
SPLIT 1: WARM-START (80/10/10 random pair split)
------------------------------------------------------------
  Train pairs: 1,263 (80.0%)
  Val   pairs:   158 (10.0%)
  Test  pairs:   158 (10.0%)

------------------------------------------------------------
SPLIT 2: CELL-COLD-START (20% cell lines held out, stratified by lineage)
------------------------------------------------------------
  Train cell lines:  53 | pairs: 1,111
  Val   cell lines:  10 | pairs:   170
  Test  cell lines:  16 | pairs:   298 (unseen)
  Leakage assertions passed

Cell-cold TEST cell lines per lineage (review before finalising CELL_COLD_FRAC):
Esophagus/Stomach    6
Bowel                5
Pancreas             3
Liver                2

If any lineage above has 0-1 test cell lines, consider raising CELL_COLD_FRAC
or accepting that lineage will have a very noisy cell-cold estimate.


In [11]:
section('Scale omics data (fit fresh on GI cell-cold TRAIN cell lines -- zero leakage)')

print(f'Fitting scalers on {len(cl_cell_train)} strictly-training GI cell lines '
      f'({len(cl_cell_train)/len(common_final)*100:.1f}% of {len(common_final)}).')

def scale_modality(df, train_ids, scaler_type, tag):
    train_mask = df.index.isin(set(train_ids))
    X_train = df.loc[train_mask].values

    if scaler_type == 'standard':
        sc = StandardScaler(); sc.fit(X_train)
    elif scaler_type == 'maxabs':
        sc = MaxAbsScaler(); sc.fit(X_train)
    else:
        sc = None

    scaled_vals = sc.transform(df.values) if sc is not None else df.values.copy()
    out = pd.DataFrame(scaled_vals.astype(np.float32), index=df.index, columns=df.columns)

    train_vals = out.loc[train_mask].values
    print(f'  [{tag:<4}] {scaler_type:<10} | train mean={train_vals.mean():+.4f} '
          f'std={train_vals.std():.4f}')
    return out, sc

mrna_scaled, sc_mrna = scale_modality(mrna_df, cl_cell_train, 'standard', 'mRNA')
cnv_scaled,  sc_cnv  = scale_modality(cnv_df,  cl_cell_train, 'maxabs',   'CNV')
mut_scaled,  _       = scale_modality(mut_df,  cl_cell_train, 'none',     'Mut')
meth_scaled, _       = scale_modality(meth_df, cl_cell_train, 'none',     'Meth')

del mrna_df, cnv_df, mut_df, meth_df
gc.collect()
print('\nUnscaled dataframes freed. Scaling complete.')



  Scale omics data (fit fresh on GI cell-cold TRAIN cell lines -- zero leakage)
Fitting scalers on 53 strictly-training GI cell lines (67.1% of 79).
  [mRNA] standard   | train mean=-0.0000 std=0.9677
  [CNV ] maxabs     | train mean=-0.4392 std=0.2736
  [Mut ] none       | train mean=+0.0271 std=0.1625
  [Meth] none       | train mean=+0.4868 std=0.2301

Unscaled dataframes freed. Scaling complete.


In [12]:
!pip install rdkit --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 55.1 MB/s eta 0:00:00


In [13]:
section('Save all outputs')
import joblib
OUT = CFG['OUT_DIR']

mrna_scaled.to_parquet(f'{OUT}ccle_mrna_gi_scaled.parquet')
cnv_scaled.to_parquet( f'{OUT}ccle_cnv_gi_scaled.parquet')
mut_scaled.to_parquet( f'{OUT}ccle_mut_gi_scaled.parquet')
meth_scaled.to_parquet(f'{OUT}ccle_meth_gi_scaled.parquet')
print('Saved: ccle_{mrna,cnv,mut,meth}_gi_scaled.parquet')

split_files = {
    'split_gi_warm_tr': ic50_warm_tr, 'split_gi_warm_vl': ic50_warm_vl, 'split_gi_warm_te': ic50_warm_te,
    'split_gi_cc_tr':   ic50_cc_tr,   'split_gi_cc_vl':   ic50_cc_vl,   'split_gi_cc_te':   ic50_cc_te,
}
for fname, df_split in split_files.items():
    df_split.to_parquet(f'{OUT}{fname}.parquet', index=False)
print('Saved: 6 IC50 split parquets (warm/cc x tr/vl/te)')

joblib.dump(sc_mrna, f'{OUT}scaler_mrna_gi.pkl')
joblib.dump(sc_cnv,  f'{OUT}scaler_cnv_gi.pkl')
print('Saved: scaler_mrna_gi.pkl, scaler_cnv_gi.pkl (fit fresh on GI CCLE data)')

# =============================================================================
# FIXED: drug_morgan_fps.npy / drug_morgan_fp_order.json don't exist in this
# dataset version. Compute Morgan ECFP4 fingerprints directly from
# drug_smiles.json (which IS present) for just the GI drug subset --
# same method as the original pipeline's Cell 16, scoped to fewer drugs.
# =============================================================================
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

with open(f"{CFG['GDSC2_BASE']}/drug_smiles.json") as f:
    smiles_map = json.load(f)

FP_RADIUS, FP_NBITS = 2, 1024   # ECFP4, 1024 bits -- same as original pipeline
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)

gi_drugs = sorted(ic50['DRUG_NAME'].unique())
print(f'\nComputing ECFP4 fingerprints for {len(gi_drugs)} GI drugs...')

fps_dict, failed = {}, []
for drug in gi_drugs:
    smiles = smiles_map.get(drug)
    if not smiles:
        failed.append(drug)
        continue
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            failed.append(drug)
            continue
        fp = morgan_gen.GetFingerprintAsNumPy(mol).astype(np.float32)
        fps_dict[drug] = fp
    except Exception:
        failed.append(drug)

print(f'  Computed: {len(fps_dict)}/{len(gi_drugs)} drugs')
if failed:
    print(f'  Failed (no SMILES or unparseable): {failed}')

gi_drugs_with_fp = sorted(fps_dict.keys())
gi_fp_matrix = np.array([fps_dict[d] for d in gi_drugs_with_fp], dtype=np.float32)

np.save(f'{OUT}gi_drug_morgan_fps.npy', gi_fp_matrix)
with open(f'{OUT}gi_drug_morgan_fp_order.json', 'w') as f:
    json.dump(gi_drugs_with_fp, f, indent=2)
print(f'Saved: gi_drug_morgan_fps.npy {gi_fp_matrix.shape}, gi_drug_morgan_fp_order.json '
      f'({len(gi_drugs_with_fp)} drugs)')

gi_drug2idx = {d: i for i, d in enumerate(gi_drugs_with_fp)}
with open(f'{OUT}gi_drug2idx.json', 'w') as f:
    json.dump(gi_drug2idx, f)
pd.Series(gi_drugs_with_fp).to_csv(f'{OUT}gi_valid_drug_names.csv', index=False, header=False)
print('Saved: gi_drug2idx.json, gi_valid_drug_names.csv')

print('\n' + '=' * 65)
print('  FINAL GI STAGE 2 DATASET SUMMARY')
print('=' * 65)
print(f'  GI lineages:        {CFG["GI_LINEAGES_KEEP"]}')
print(f'  Cell lines:          {len(common_final)}')
print(f'  Drugs (with FP):     {len(gi_drugs_with_fp)}')
print(f'  mRNA/CNV/Mut/Meth:   {mrna_scaled.shape[1]}/{cnv_scaled.shape[1]}/'
      f'{mut_scaled.shape[1]}/{meth_scaled.shape[1]} genes (GIBridge-aligned)')
print(f'  WARM-START:      Train {len(ic50_warm_tr):>5,} | Val {len(ic50_warm_vl):>4,} | '
      f'Test {len(ic50_warm_te):>4,}')
print(f'  CELL-COLD-START: Train {len(ic50_cc_tr):>5,} | Val {len(ic50_cc_vl):>4,} | '
      f'Test {len(ic50_cc_te):>4,}')
print('=' * 65)
print('GI Stage 2 data processing complete.')


  Save all outputs
Saved: ccle_{mrna,cnv,mut,meth}_gi_scaled.parquet
Saved: 6 IC50 split parquets (warm/cc x tr/vl/te)
Saved: scaler_mrna_gi.pkl, scaler_cnv_gi.pkl (fit fresh on GI CCLE data)

Computing ECFP4 fingerprints for 71 GI drugs...
  Computed: 71/71 drugs
Saved: gi_drug_morgan_fps.npy (71, 1024), gi_drug_morgan_fp_order.json (71 drugs)
Saved: gi_drug2idx.json, gi_valid_drug_names.csv

  FINAL GI STAGE 2 DATASET SUMMARY
  GI lineages:        ['Bowel', 'Esophagus/Stomach', 'Pancreas', 'Liver']
  Cell lines:          79
  Drugs (with FP):     71
  mRNA/CNV/Mut/Meth:   4000/1750/1250/3000 genes (GIBridge-aligned)
  WARM-START:      Train 1,263 | Val  158 | Test  158
  CELL-COLD-START: Train 1,111 | Val  170 | Test  298
GI Stage 2 data processing complete.
